In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from numpy.f2py.symbolic import as_ge

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_PATH = RAW_DATA_DIR / "train.csv"
TEST_PATH = RAW_DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = RAW_DATA_DIR / "sample_submission.csv"

train_df = pl.read_csv(TRAIN_PATH)
test_df = pl.read_csv(TEST_PATH)
sample_submission_df = pl.read_csv(SAMPLE_SUBMISSION_PATH)

train_df.shape, test_df.shape, sample_submission_df.shape

((699635, 23), (299844, 22), (299844, 2))

In [16]:
display(train_df.head(5))
print()
print("AMOUNT OF ROWS AND COLUMNS: ")
display(train_df.shape)
print()
print("ALL NAMES OF COLUMNS: ")
display(train_df.columns)
print()
display(train_df.schema)

id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,str,str,str,str,bool
0,36,694,4,4,4,3,1,4,1,1,3,4,4,3,1,0,0.0,"""Female""","""Loyal Customer""","""Personal Travel""","""Eco""",false
1,56,651,5,5,5,5,5,4,4,5,5,5,5,4,3,0,0.0,"""Male""","""Loyal Customer""","""Business travel""","""Business""",true
2,31,1371,3,4,2,5,3,3,3,3,4,3,4,5,3,0,0.0,"""Female""","""Loyal Customer""","""Personal Travel""","""Eco""",false
3,10,1616,3,5,3,3,3,3,3,3,4,5,3,4,3,0,0.0,"""Male""","""Loyal Customer""","""Personal Travel""","""Eco""",false
4,23,481,3,4,3,4,5,3,5,5,5,4,5,3,5,5,0.0,"""Female""","""disloyal Customer""","""Business travel""","""Eco""",false



AMOUNT OF ROWS AND COLUMNS: 


(699635, 23)


ALL NAMES OF COLUMNS: 


['id',
 'Age',
 'Flight Distance',
 'Inflight wifi service',
 'Departure/Arrival time convenient',
 'Ease of Online booking',
 'Gate location',
 'Food and drink',
 'Online boarding',
 'Seat comfort',
 'Inflight entertainment',
 'On-board service',
 'Leg room service',
 'Baggage handling',
 'Checkin service',
 'Cleanliness',
 'Departure Delay in Minutes',
 'Arrival Delay in Minutes',
 'Gender',
 'Customer Type',
 'Type of Travel',
 'Class',
 'satisfaction']

Schema([('id', Int64),
        ('Age', Int64),
        ('Flight Distance', Int64),
        ('Inflight wifi service', Int64),
        ('Departure/Arrival time convenient', Int64),
        ('Ease of Online booking', Int64),
        ('Gate location', Int64),
        ('Food and drink', Int64),
        ('Online boarding', Int64),
        ('Seat comfort', Int64),
        ('Inflight entertainment', Int64),
        ('On-board service', Int64),
        ('Leg room service', Int64),
        ('Baggage handling', Int64),
        ('Checkin service', Int64),
        ('Cleanliness', Int64),
        ('Departure Delay in Minutes', Int64),
        ('Arrival Delay in Minutes', Float64),
        ('Gender', String),
        ('Customer Type', String),
        ('Type of Travel', String),
        ('Class', String),
        ('satisfaction', Boolean)])

In [24]:
display(train_df.select(pl.all().n_unique()))
print()
display(train_df.null_count())
print()

for i in train_df.columns:
    if train_df[i].n_unique() <= 6:
        print(train_df[i].unique())
        print(train_df[i].value_counts())

id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
699635,75,3474,6,6,6,6,6,6,6,6,6,6,5,6,6,167,169,2,2,2,3,2


id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,292,0,0,0,0,0



shape: (6,)
Series: 'Inflight wifi service' [i64]
[
	0
	1
	2
	3
	4
	5
]
shape: (6, 2)
┌───────────────────────┬────────┐
│ Inflight wifi service ┆ count  │
│ ---                   ┆ ---    │
│ i64                   ┆ u32    │
╞═══════════════════════╪════════╡
│ 4                     ┆ 128136 │
│ 3                     ┆ 184436 │
│ 1                     ┆ 107739 │
│ 2                     ┆ 186870 │
│ 0                     ┆ 13102  │
│ 5                     ┆ 79352  │
└───────────────────────┴────────┘
shape: (6,)
Series: 'Departure/Arrival time convenient' [i64]
[
	0
	1
	2
	3
	4
	5
]
shape: (6, 2)
┌─────────────────────────────────┬────────┐
│ Departure/Arrival time conveni… ┆ count  │
│ ---                             ┆ ---    │
│ i64                             ┆ u32    │
╞═════════════════════════════════╪════════╡
│ 1                               ┆ 95072  │
│ 4                               ┆ 181530 │
│ 2                               ┆ 118709 │
│ 0                               ┆